# Hipótesis
Los resultados de la Fase A mostraron que:

El decoder representa solo ~9 % de los parámetros totales.
El IoU Prediction Head sobreestima la calidad real de las máscaras.
El mecanismo de atención prompt→imagen ocurre dentro del decoder.

Hipótesis:

Las representaciones aprendidas por el decoder en SA‑1B no están totalmente adaptadas a estructuras parcelarias observadas en Sentinel‑2. Por ello, ajustar el último bloque del TwoWayTransformer y el IoU Prediction Head permitirá adaptar la toma de decisiones sin modificar el encoder.

In [10]:
# URL de repositorio
!git clone https://github.com/ZagonTM/Sen4Sam

# Una vez clonado, nos movemos al directorio del repositorio o lo añadimos al path
import os
if os.path.exists('Sen4Sam'):
    %cd Sen4Sam
    print("Acceso al repositorio Sen4Sam exitoso.")
else:
    print("No se pudo encontrar la carpeta del repositorio. Verifica la URL de clonación.")

Cloning into 'Sen4Sam'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 104 (delta 36), reused 89 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 1.55 MiB | 18.84 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/Sen4Sam
Acceso al repositorio Sen4Sam exitoso.


In [12]:
# Instalar Segment Anything 2 (SAM 2) y sus dependencias
!pip install -q git+https://github.com/facebookresearch/sam2.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [14]:
# Crear directorio de checkpoints y descargar los pesos del modelo
!mkdir -p checkpoints
!wget -P checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt

--2026-11-04 15:26:36--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 54.230.79.93, 54.230.79.88, 54.230.79.122, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|54.230.79.93|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 184416285 (176M) [application/vnd.snesdev-page-table]
Saving to: ‘checkpoints/sam2.1_hiera_small.pt’

sam2.1_hiera_small. 100%[===================>] 175.87M   242MB/s    in 0.7s    

2026-11-04 15:26:37 (242 MB/s) - ‘checkpoints/sam2.1_hiera_small.pt’ saved [184416285/184416285]



**Cargamos modelo**

In [15]:
import sys
from pathlib import Path

ROOT = Path.cwd()

sys.path.insert(0, str(ROOT / "src"))

from sam2_auto_segment import build_model

model = build_model()

**Contamos parámetros del decoder**

In [16]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

decoder = model.sam_mask_decoder

print(
    "decoder:",
    count_params(decoder)/1e6,
    "M"
)

decoder: 4.215109 M


**Medir el IoU Head**

In [17]:
iou_head = decoder.iou_prediction_head

print(
    "iou head:",
    count_params(iou_head)
)

iou head: 132612


**Medir la última capa del transformer**

Tenemos:

In [18]:
decoder.transformer.layers[0]

decoder.transformer.layers[1]

TwoWayAttentionBlock(
  (self_attn): Attention(
    (q_proj): Linear(in_features=256, out_features=256, bias=True)
    (k_proj): Linear(in_features=256, out_features=256, bias=True)
    (v_proj): Linear(in_features=256, out_features=256, bias=True)
    (out_proj): Linear(in_features=256, out_features=256, bias=True)
  )
  (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (cross_attn_token_to_image): Attention(
    (q_proj): Linear(in_features=256, out_features=128, bias=True)
    (k_proj): Linear(in_features=256, out_features=128, bias=True)
    (v_proj): Linear(in_features=256, out_features=128, bias=True)
    (out_proj): Linear(in_features=128, out_features=256, bias=True)
  )
  (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (mlp): MLP(
    (layers): ModuleList(
      (0): Linear(in_features=256, out_features=2048, bias=True)
      (1): Linear(in_features=2048, out_features=256, bias=True)
    )
    (act): ReLU()
  )
  (norm3): LayerNorm((256,), eps=1

Medimos:

In [19]:
last_block = decoder.transformer.layers[1]

print(
    count_params(last_block)
)

1579520


In [20]:
import pandas as pd

rows = [
    {
        "component":"iou_head",
        "params":count_params(iou_head)
    },

    {
        "component":"last_transformer_block",
        "params":count_params(last_block)
    },

    {
        "component":"full_decoder",
        "params":count_params(decoder)
    }
]

pd.DataFrame(rows)

,component,params
0,iou_head,132612
1,last_transformer_block,1579520
2,full_decoder,4215109


**Congelar todo**

In [21]:
for p in model.parameters():

    p.requires_grad = False

**Descongelar sólo la intervención**

In [22]:
for p in iou_head.parameters():

    p.requires_grad = True

for p in last_block.parameters():

    p.requires_grad = True

**Cuántos parámetros entrenaremos**

In [24]:
trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters: {total:,}")
print(f"Fraction trainable: {100*trainable/total:.2f}%")

Trainable parameters: 1,712,132
Total parameters: 46,060,354
Fraction trainable: 3.72%


In [25]:
import pandas as pd

intervention = pd.DataFrame([
    ["Image Encoder", "34.3 M", "Frozen"],
    ["Transformer Block 0", "1.58 M", "Frozen"],
    ["Transformer Block 1", "1.58 M", "Trainable"],
    ["IoU Prediction Head", "132 K", "Trainable"],
])

intervention.columns = ["Component","Parameters","Status"]

intervention

,Component,Parameters,Status
0,Image Encoder,34.3 M,Frozen
1,Transformer Block 0,1.58 M,Frozen
2,Transformer Block 1,1.58 M,Trainable
3,IoU Prediction Head,132 K,Trainable


# Comparación con la línea base
Baseline:


*   Encoder congelado
*   Decoder congelado
*   0 parámetros entrenables


Intervención preliminar:

* Encoder congelado

* Transformer Layer 1 entrenable

* IoU Prediction Head entrenable

* 3.72 % de parámetros entrenables

# Motivación de la intervención
Durante la Fase A observamos que el decoder asignó una puntuación IoU = 0.703 a una máscara cuya IoU real contra la parcela de referencia fue 0.301.

Esta discrepancia sugiere que la estimación de calidad aprendida en imágenes naturales puede estar mal calibrada para imágenes Sentinel‑2.

Por ello la intervención incluye el IoU Prediction Head.

# Próximos pasos

1. Construir un conjunto de entrenamiento utilizando máscaras generadas por SAM 2 y parcelas de Sen4AgriNet.
2. Calcular IoU real para cada máscara.

3. Fine-tuning del último bloque del decoder y del IoU Prediction Head.

4. Comparación contra la línea base mediante:

* Mean Best IoU
* Median Best IoU
* Fraction IoU > 0.5
* Mask-on-field fraction